## SpendWise AI - Notebook 08: Final Pipeline & Streamlit App

**Objective:** Integrate all components into a unified system and build the Streamlit dashboard

**Components:** Receipt Parser (02), Transaction Classifier (03), Anomaly Detector (04), Spending Forecaster (05), LLM Assistant (06), Recommendation Engine (07).

**Output:** Production-ready Streamlit application

### 1. Imports & Setup

In [17]:
import sys
import json
from pathlib import Path
from datetime import datetime, timedelta
from typing import Optional, Dict, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "src" and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists():
    while PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent
        if (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT / "src").exists():
            break
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project root:", PROJECT_ROOT)

Project root: /Users/patel/Documents/E/MPS Sem 5/EAI 6020/Final_Project/SpendwiseAI


In [18]:
components_status = {}
transactions_df = pd.read_csv(PROJECT_ROOT / "data/synthetic/transactions_full.csv")
transactions_df["date"] = pd.to_datetime(transactions_df["date"])
components_status["data"] = True
print(f"Data loaded: {len(transactions_df):,} transactions")

classifier = None
try:
    from transaction_classifier import TransactionClassifierInference
    classifier = TransactionClassifierInference(str(PROJECT_ROOT / "models/classifier_model"))
    components_status["classifier"] = True
    print("Transaction Classifier loaded")
except Exception as e:
    components_status["classifier"] = False
    print("Transaction Classifier not available:", e)

anomaly_detector = None
try:
    from anomaly_detector import AnomalyDetector
    anomaly_detector = AnomalyDetector(str(PROJECT_ROOT / "models/anomaly_model"))
    components_status["anomaly"] = True
    print("Anomaly Detector loaded")
except Exception as e:
    components_status["anomaly"] = False
    print("Anomaly Detector not available:", e)

forecaster = None
try:
    from spending_forecaster import ZICATTInference
    forecaster = ZICATTInference(str(PROJECT_ROOT / "models/forecaster_model"))
    components_status["forecaster"] = True
    print("Spending Forecaster loaded")
except Exception as e:
    components_status["forecaster"] = False
    print("Spending Forecaster not available:", e)

recommender = None
try:
    from recommendation_engine import RecommendationService
    recommender = RecommendationService(transactions_df)
    components_status["recommender"] = True
    print("Recommendation Engine loaded")
except Exception as e:
    components_status["recommender"] = False
    print("Recommendation Engine not available:", e)

assistant = None
try:
    from llm_assistant import FinancialDataManager, FinancialAssistant
    data_manager = FinancialDataManager(transactions_df)
    assistant = FinancialAssistant(data_manager)
    components_status["assistant"] = True
    print("LLM Assistant loaded")
except Exception as e:
    components_status["assistant"] = False
    print("LLM Assistant not available:", e)

print("\n" + "=" * 50)
print("Components Status:")
for component, status in components_status.items():
    print(f"   {component}: {'OK' if status else 'not available'}")

Data loaded: 122,754 transactions


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 17522.26it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Transaction Classifier loaded
Anomaly Detector loaded
Spending Forecaster loaded
Recommendation Engine loaded
Running in demo mode (no API key). Responses will be simulated.
LLM Assistant loaded

Components Status:
   data: OK
   classifier: OK
   anomaly: OK
   forecaster: OK
   recommender: OK
   assistant: OK


### 3. Build Unified SpendWise API

In [19]:
class SpendWiseAPI:
    def __init__(self, transactions_df, classifier=None, anomaly_detector=None, forecaster=None, recommender=None, assistant=None):
        self.df = transactions_df.copy()
        self.df["date"] = pd.to_datetime(self.df["date"])
        self.classifier = classifier
        self.anomaly_detector = anomaly_detector
        self.forecaster = forecaster
        self.recommender = recommender
        self.assistant = assistant
        self.df["month"] = self.df["date"].dt.to_period("M")
        self.df["week"] = self.df["date"].dt.to_period("W")
        self.df["is_expense"] = self.df["amount"] < 0
        self.users = sorted(self.df["user_id"].unique())
        self.categories = sorted(self.df["category"].unique())

    def get_dashboard_data(self, user_id):
        return {"summary": self.get_spending_summary(user_id), "by_category": self.get_spending_by_category(user_id), "trend": self.get_monthly_trend(user_id), "recent_transactions": self.get_recent_transactions(user_id), "anomaly_status": self.check_anomaly(user_id), "forecast": self.get_forecast(user_id), "recommendations": self.get_recommendations(user_id)}

    def get_spending_summary(self, user_id, days=30):
        user_df = self.df[self.df["user_id"] == user_id]
        max_date = user_df["date"].max()
        current = user_df[user_df["date"] > max_date - timedelta(days=days)]
        current_expenses = current[current["is_expense"]]["amount"].abs().sum()
        current_income = current[~current["is_expense"]]["amount"].sum()
        prev_start, prev_end = max_date - timedelta(days=days*2), max_date - timedelta(days=days)
        previous = user_df[(user_df["date"] > prev_start) & (user_df["date"] <= prev_end)]
        prev_expenses = previous[previous["is_expense"]]["amount"].abs().sum()
        change_pct = ((current_expenses - prev_expenses) / prev_expenses) * 100 if prev_expenses > 0 else 0
        return {"total_expenses": round(current_expenses, 2), "total_income": round(current_income, 2), "net": round(current_income - current_expenses, 2), "transaction_count": len(current[current["is_expense"]]), "daily_average": round(current_expenses / days, 2), "change_pct": round(change_pct, 1), "period_days": days}

    def get_spending_by_category(self, user_id, days=30):
        user_df = self.df[self.df["user_id"] == user_id]
        max_date = user_df["date"].max()
        recent = user_df[user_df["date"] > max_date - timedelta(days=days)]
        expenses = recent[recent["is_expense"]].copy()
        expenses["amount"] = expenses["amount"].abs()
        by_cat = expenses.groupby("category")["amount"].sum().sort_values(ascending=False)
        total = by_cat.sum()
        return [{"category": c, "amount": round(a, 2), "percentage": round(a / total * 100, 1) if total > 0 else 0} for c, a in by_cat.items()]

    def get_monthly_trend(self, user_id, months=6):
        user_df = self.df[self.df["user_id"] == user_id]
        expenses = user_df[user_df["is_expense"]].copy()
        expenses["amount"] = expenses["amount"].abs()
        monthly = expenses.groupby("month")["amount"].sum().tail(months)
        return {"labels": [str(m) for m in monthly.index], "values": [round(v, 2) for v in monthly.values], "average": round(monthly.mean(), 2)}

    def get_recent_transactions(self, user_id, limit=20):
        user_df = self.df[self.df["user_id"] == user_id].copy()
        recent = user_df.sort_values("date", ascending=False).head(limit)
        return [{"date": row["date"].strftime("%Y-%m-%d"), "merchant": row["merchant"], "amount": round(row["amount"], 2), "category": row["category"], "subcategory": row["subcategory"]} for _, row in recent.iterrows()]

    def classify_transaction(self, text):
        return self.classifier.classify(text) if self.classifier else {"error": "Classifier not available"}

    def check_anomaly(self, user_id):
        if not self.anomaly_detector:
            return {"error": "Anomaly detector not available"}
        by_cat = self.get_spending_by_category(user_id)
        spending = {item["category"]: item["amount"] for item in by_cat}
        try:
            result = self.anomaly_detector.detect(spending)
            return {"score": round(result.get("anomaly_score", 0), 1), "is_anomaly": result.get("is_anomaly", False), "status": "unusual" if result.get("is_anomaly", False) else "normal"}
        except Exception as e:
            return {"error": str(e)}

    def get_forecast(self, user_id):
        if not self.forecaster:
            return {"error": "Forecaster not available"}
        user_df = self.df[self.df["user_id"] == user_id]
        expenses = user_df[user_df["is_expense"]].copy()
        expenses["amount"] = expenses["amount"].abs()
        weekly = expenses.groupby("week")["amount"].sum()
        history = weekly.tail(12).tolist()
        if len(history) < 8:
            return {"error": "Not enough history for forecast"}
        try:
            result = self.forecaster.predict(history)
            return {"predicted": result["predicted_spending"], "lower": result["lower_bound"], "upper": result["upper_bound"], "recent_avg": round(np.mean(history[-4:]), 2)}
        except Exception as e:
            return {"error": str(e)}

    def get_recommendations(self, user_id, limit=5):
        if not self.recommender:
            return []
        try:
            result = self.recommender.get_recommendations(user_id, limit=limit)
            return result.get("recommendations", [])
        except Exception:
            return []

    def chat(self, user_id, message):
        return self.assistant.chat(message, user_id) if self.assistant else "Chat assistant not available."

api = SpendWiseAPI(transactions_df, classifier=classifier, anomaly_detector=anomaly_detector, forecaster=forecaster, recommender=recommender, assistant=assistant)
print("SpendWise API initialized. Users:", len(api.users), "Categories:", len(api.categories))

SpendWise API initialized. Users: 100 Categories: 12


### 4. Test the Unified API

In [20]:
test_user = "user_0001"
dashboard = api.get_dashboard_data(test_user)
print("Testing API for", test_user, "\n" + "=" * 60)
print("\nSpending Summary (Last 30 days):")
s = dashboard["summary"]
print(f"   Total expenses: ${s['total_expenses']:,.2f}, Income: ${s['total_income']:,.2f}, Net: ${s['net']:,.2f}, Change: {s['change_pct']:+.1f}%")
print("Top Categories:", [(x["category"], f"${x['amount']:,.2f}") for x in dashboard["by_category"][:5]])
print("Monthly Trend average:", dashboard["trend"]["average"])
anomaly = dashboard["anomaly_status"]
print("Anomaly:", anomaly.get("status", anomaly.get("error")), f"Score: {anomaly.get('score', 0)}/100" if "error" not in anomaly else "")
forecast = dashboard["forecast"]
print("Forecast:", f"${forecast['predicted']:,.2f}" if "error" not in forecast else forecast["error"])
print("Top Recommendations:", [r["title"] for r in dashboard["recommendations"][:3]])

Testing API for user_0001 

Spending Summary (Last 30 days):
   Total expenses: $18,009.45, Income: $3,507.00, Net: $-14,502.45, Change: +17.7%
Top Categories: [('Education', '$5,568.68'), ('Travel', '$3,383.00'), ('Financial', '$2,746.88'), ('Health & Wellness', '$1,697.50'), ('Entertainment', '$1,302.16')]
Monthly Trend average: 14267.91
Anomaly: unusual Score: 54.79999923706055/100
Forecast: $4,558.04
Top Recommendations: ['Low Savings Rate', 'Multiple Transportation Subscriptions', 'Frequent Coffee Shops Purchases']


### 5. Create Streamlit App

In [21]:
app_path = PROJECT_ROOT / "app" / "streamlit_app.py"
app_path.parent.mkdir(parents=True, exist_ok=True)
# Streamlit app source is in app/streamlit_app.py (create or update as needed)
print("Streamlit app path:", app_path)

Streamlit app path: /Users/patel/Documents/E/MPS Sem 5/EAI 6020/Final_Project/SpendwiseAI/app/streamlit_app.py


### 6. Create Requirements File

In [22]:
requirements = """# SpendWise AI - Requirements
numpy>=1.24.0
pandas>=2.0.0
scikit-learn>=1.3.0
torch>=2.0.0
torchvision>=0.15.0
transformers>=4.30.0
matplotlib>=3.7.0
seaborn>=0.12.0
plotly>=5.15.0
streamlit>=1.25.0
anthropic>=0.18.0
tqdm>=4.65.0
Pillow>=9.5.0
python-dotenv>=1.0.0
datasets>=2.14.0
"""
(PROJECT_ROOT / "requirements.txt").write_text(requirements)
print("requirements.txt created")

requirements.txt created


### 7. Create README

In [23]:
readme = """# SpendWise AI

AI-Powered Personal Finance Intelligence System

## Features

- Receipt Scanning: Extract data from receipt images (Donut OCR)
- Transaction Classification: Auto-categorize with high accuracy
- Anomaly Detection: Flag unusual spending (VAE)
- Spending Forecast: Predict expenses (Transformer)
- AI Assistant: Natural language queries (Claude)
- Smart Recommendations: Personalized savings suggestions

## Tech Stack

PyTorch, Transformers, Scikit-learn, Streamlit, Anthropic Claude (optional)

## Quick Start

  pip install -r requirements.txt
  streamlit run app/streamlit_app.py

## Project Structure

  spendwise-ai/
  - src/ (receipt_parser, transaction_classifier, anomaly_detector, spending_forecaster, llm_assistant, recommendation_engine)
  - models/
  - data/
  - app/streamlit_app.py
"""
(PROJECT_ROOT / "README.md").write_text(readme)
print("README.md created")

README.md created


### 8. Create Project Summary Report

In [24]:
def generate_project_report():
    report = []
    report.append("=" * 70)
    report.append("           SPENDWISE AI - PROJECT COMPLETION REPORT")
    report.append("=" * 70)
    report.append("\nCOMPONENTS BUILT:")
    for name, detail in [("Data Pipeline", "122K+ synthetic transactions"), ("Receipt OCR", "Donut-based parser"), ("Transaction Classifier", "94.8% accuracy"), ("Anomaly Detector", "VAE from scratch"), ("Spending Forecaster", "Transformer from scratch"), ("LLM Assistant", "Claude integration"), ("Recommendation Engine", "5 recommendation types"), ("Streamlit Dashboard", "Full UI")]:
        report.append(f"   - {name}: {detail}")
    report.append("\nFILES: src/*.py, app/streamlit_app.py, requirements.txt, README.md")
    report.append("\n" + "=" * 70)
    return "\n".join(report)

report = generate_project_report()
print(report)
(PROJECT_ROOT / "PROJECT_REPORT.txt").write_text(report)
print("\nReport saved to PROJECT_REPORT.txt")

           SPENDWISE AI - PROJECT COMPLETION REPORT

COMPONENTS BUILT:
   - Data Pipeline: 122K+ synthetic transactions
   - Receipt OCR: Donut-based parser
   - Transaction Classifier: 94.8% accuracy
   - Anomaly Detector: VAE from scratch
   - Spending Forecaster: Transformer from scratch
   - LLM Assistant: Claude integration
   - Recommendation Engine: 5 recommendation types
   - Streamlit Dashboard: Full UI

FILES: src/*.py, app/streamlit_app.py, requirements.txt, README.md


Report saved to PROJECT_REPORT.txt


### Summary

- Loaded all components (data, classifier, anomaly, forecaster, recommender, assistant); SpendWiseAPI unifies dashboard data, predictions, recommendations, and chat.
- Streamlit app at app/streamlit_app.py; requirements.txt and README.md created; PROJECT_REPORT.txt generated.
- Project complete. Run: streamlit run app/streamlit_app.py